In [ ]:
# =============================================================================
# CAMADA SILVER - CARTAS - MAGIC: THE GATHERING
# =============================================================================
"""
Script Python para processamento da tabela TB_FATO_CARTAS.
Transformação e limpeza de dados da Bronze para Silver.

CLASSIFICAÇÃO DAMA-DMBOK (#116): Fato - uma linha por impressão de carta
(grão), com medidas quantitativas (Vlr_usd/Vlr_eur/Vlr_tix, Qtd_custo_mana,
Qtd_cores) e chaves estrangeiras implícitas pra dimensões (Cod_colecao ->
TB_DIM_COLECOES). Daí o prefixo TB_FATO_ e o nome sem o segmento redundante
"SILVER" (já implícito no schema silver.* do Unity Catalog).

CHAVE ÚNICA: Id_carta + Dt_ingestao_preco (ver save_silver_table no fim do
notebook). Dt_ingestao_preco pode ser NULO (carta sem preço encontrado em
attach_prices) - por isso o Unity Catalog só grava um comentário de tabela
sinalizando a chave; a constraint PRIMARY KEY (que exige colunas NOT NULL) é
tentada best-effort em save_to_silver e degrada pro comentário quando falha
(ver silver_utils.py).

CONVENÇÃO DE NOME/CASE DE COLUNA: prefixo semântico já usado no projeto
(Id_/Nme_/Desc_/Cod_/Dt_/Qtd_/Vlr_/Num_/Url_) + primeira letra maiúscula,
resto minúsculo, sem acento - todas as colunas 100% PT-BR a partir da Silver
(pedido do usuário; Bronze/Ingestion continuam passthrough 1:1 da fonte).

USO DE SILVER_UTILS.PY:
- Centralização de funções comuns
- Padronização de processamento
- Redução de código duplicado

TRANSFORMAÇÃO DE NEGÓCIO EM SQL:
- Toda a lógica de limpeza/derivação roda via spark.sql() sobre temp views,
  em vez de encadear .withColumn() no DataFrame API.
- Cada view representa um estágio da transformação.

ESTÁGIO 0 (PADRONIZAÇÃO DE NOMES) - AUD-20 (#135) / #115:
- Bronze é passthrough 1:1 da Scryfall/legado (id, name, manaCost, set...).
  O Estágio 0 faz o SELECT explícito de toda coluna da Bronze cards pro nome
  PT-BR final (ver CONVENÇÃO acima) - nenhuma coluna sobra sem tradução.
- Correção de bug (achado nesta revisão, não fazia parte do pedido original):
  os estágios seguintes (herdados) tinham um bloco de fallback que checava
  `nome_pt_br in df.columns`, onde `df` é o DataFrame CRU da Bronze (colunas
  em inglês/camelCase). Essa checagem NUNCA era verdadeira (ex.: "NME_CARD"
  nunca está em ["id","name","manaCost",...]), então TODA coluna de negócio
  (nome, artista, raridade, tipo, custo de mana...) caía sempre no fallback
  NULL, silenciosamente, em toda execução. Como o Estágio 0 agora garante
  (via CARDS_SCHEMA da Ingestion) que toda coluna renomeada sempre existe, o
  bloco de fallback foi removido - ele resolvia um schema-drift que o
  contrato da Ingestion já impede, e escondia esse bug em vez de proteger
  contra ele. Único fallback condicional mantido: Id_oracle (oracle_id é
  novo - #135 - pode faltar em partição gravada antes da mudança).

UNIFICAÇÃO COM CARDPRICES:
- Cards e preços vêm de fontes diferentes (MTG API vs Scryfall) e o preço só
  existe por NOME (Scryfall é consultado por /cards/named?exact=<name>, sem
  granularidade de impressão) - não há Id_carta do lado do preço. O join,
  portanto, é feito aqui na Silver por nome, e uma única linha de preço se
  propaga para todas as impressões (Id_carta) daquele nome.

RESOLUÇÃO DE MIGRAÇÃO DE ID (AUD-20 - #135):
- A Scryfall às vezes funde duas impressões (merge) ou remove uma (delete),
  trocando o scryfall_id - ver GET /migrations, ingerido na Bronze migrations.
  Id_scryfall_canonico resolve a cadeia de merges até o id final e é anexado
  como coluna nova, sem reatribuir Id_carta (identificador da própria
  impressão, preservado como veio da fonte - restrição do prompt de "não
  alterar IDs/códigos"). Gold usa Id_scryfall_canonico nas window functions
  no lugar do Id_carta bruto quando precisar agrupar por carta através de uma
  migração.

REGRA "SEM ( ) { } NO DADO SILVER" (pedido do usuário):
- Texto de carta/custo de mana/legalidades vêm da Scryfall com notação de
  símbolo entre chaves (ex.: "{2}{U}{U}") e texto de lembrete entre
  parênteses (ex.: "(Add one mana of any color.)"), e legalities é um dict
  serializado. Todos convertidos pra notação com colchetes ([...]) no
  Estágio 3 - símbolos comuns viram um rótulo legível (ex.: "[White]"), o
  resto (custo genérico, mana híbrida/phyrexiana, loyalty, parênteses) usa um
  catch-all genérico que preserva o conteúdo trocando só o delimitador.
"""

# =============================================================================
# BIBLIOTECAS UTILIZADAS
# =============================================================================
import logging
from pyspark.sql.functions import col

# =============================================================================
# CARREGAMENTO DO MÓDULO UTILITÁRIO
# =============================================================================
# Importar infraestrutura comum e funções do silver_utils usando %run (Databricks)
%run "../../00 - Common/Dev/base_utils"
%run ./silver_utils

# =============================================================================
# CONFIGURAÇÃO INICIAL
# =============================================================================
def setup_logging():
    """Configura logging para o script"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    return logging.getLogger(__name__)

def transform_cards_silver(df):
    """
    Transformação específica para tabela Cartas, via SQL (spark.sql sobre temp views)
    """
    if not df:
        return None

    logger = logging.getLogger(__name__)
    logger.info("Iniciando transformações específicas para Cartas...")

    df.createOrReplaceTempView("_cards_bronze")

    # Estágio 0: SELECT explícito Bronze crua -> nome PT-BR final (ver
    # docstring do módulo). oracle_id é a única coluna aqui que pode não
    # existir ainda em partições antigas da Bronze (capturada a partir de
    # #135 na Ingestion) - fallback NULL tipado, sem quebrar o resto do
    # pipeline; todas as outras colunas vêm do CARDS_SCHEMA da Ingestion e
    # sempre existem (valor pode ser NULL, a coluna nunca falta).
    if "oracle_id" in df.columns:
        oracle_id_select = "oracle_id AS Id_oracle"
    else:
        logger.warning("Coluna oracle_id ausente na Bronze cards - Id_oracle ficará NULL (ver #135).")
        oracle_id_select = "CAST(NULL AS STRING) AS Id_oracle"

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW _cards_stage0 AS
        SELECT
            id AS Id_carta,
            {oracle_id_select},
            name AS Nme_carta,
            manaCost AS Desc_custo_mana,
            cmc AS Qtd_custo_mana,
            colors AS Cod_cores,
            colorIdentity AS Cod_identidade_cor,
            type AS Nme_tipo_carta,
            types AS Desc_tipos,
            subtypes AS Desc_subtipos,
            rarity AS Nme_raridade,
            `set` AS Cod_colecao,
            setName AS Nme_colecao,
            text AS Desc_carta,
            artist AS Nme_artista,
            number AS Num_colecionador,
            power AS Nme_forca,
            toughness AS Nme_resistencia,
            layout AS Nme_disposicao_carta,
            multiverseid AS Id_multiverso,
            imageUrl AS Url_imagem,
            variations AS Cod_variacoes,
            foreignNames AS Desc_nomes_estrangeiros,
            printings AS Desc_impressoes,
            originalText AS Desc_carta_original,
            originalType AS Nme_tipo_original,
            legalities AS Desc_legalidades,
            ingestion_timestamp AS Dt_ingestao,
            source AS Nme_fonte,
            endpoint AS Desc_url_origem,
            source_file AS Desc_arquivo_origem,
            bronze_run_id AS Id_execucao_bronze,
            bronze_ingestion_timestamp AS Dt_ingestao_bronze
        FROM _cards_bronze
    """)

    # Estágio 1: filtro temporal (últimos 5 anos). Dt_ingestao sempre existe
    # (coluna técnica obrigatória da Bronze) - sem fallback aqui: um NULL
    # nela zeraria silenciosamente o filtro (WHERE NULL >= ...) e descartaria
    # o lote inteiro sem erro, pior que um crash.
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW _cards_stage1 AS
        SELECT *
        FROM _cards_stage0
        WHERE Dt_ingestao >= add_months(current_date(), -60)
    """)

    # Estágio 2: limpeza/derivação de negócio. \\[ \\] no literal SQL: Spark
    # desfaz um backslash simples antes de um caractere sem escape
    # reconhecido (aqui viraria '[|]|"', uma regex válida mas errada - classe
    # de caracteres, não escape literal). Dobrar o backslash na fonte Python
    # garante que sobra um só depois do unescaping do Spark.
    spark.sql(r"""
        CREATE OR REPLACE TEMP VIEW _cards_stage2 AS
        SELECT
            * EXCEPT (Nme_carta, Nme_artista, Nme_raridade, Nme_colecao, Desc_carta,
                      Desc_custo_mana, Qtd_custo_mana, Nme_forca, Nme_resistencia,
                      Cod_colecao, Desc_impressoes, Cod_variacoes, Cod_cores,
                      Cod_identidade_cor, Desc_subtipos, Desc_tipos, Nme_tipo_carta,
                      Dt_ingestao),

            initcap(trim(Nme_carta)) AS Nme_carta,
            initcap(trim(Nme_artista)) AS Nme_artista,
            initcap(trim(Nme_raridade)) AS Nme_raridade,
            initcap(trim(Nme_colecao)) AS Nme_colecao,
            CASE WHEN Desc_carta IS NULL OR Desc_carta = '' THEN 'NA' ELSE trim(Desc_carta) END AS Desc_carta,
            CASE WHEN Desc_custo_mana IS NULL OR Desc_custo_mana = '' THEN 'NA' ELSE trim(Desc_custo_mana) END AS Desc_custo_mana,
            coalesce(Qtd_custo_mana, 0) AS Qtd_custo_mana,
            -- Nme_forca/Nme_resistencia são STRING na Bronze e podem legitimamente
            -- valer "*", "1+*" etc. (poder/resistência variável - ex.: Tarmogoyf).
            -- Fallback como string ('0'), não int: coalesce(STRING_COL, 0) força
            -- um implicit cast pra BIGINT, que quebra (CAST_INVALID_INPUT) no
            -- primeiro valor não-numérico.
            coalesce(Nme_forca, '0') AS Nme_forca,
            coalesce(Nme_resistencia, '0') AS Nme_resistencia,
            upper(Cod_colecao) AS Cod_colecao,
            regexp_replace(Desc_impressoes, '\\[|\\]|"', '') AS Desc_impressoes,
            regexp_replace(Cod_variacoes, '\\[|\\]|"', '') AS Cod_variacoes,
            regexp_replace(Cod_cores, '\\[|\\]|"', '') AS Cod_cores,
            regexp_replace(Cod_identidade_cor, '\\[|\\]|"', '') AS Cod_identidade_cor,
            regexp_replace(Desc_subtipos, '\\[|\\]|"', '') AS Desc_subtipos,
            CASE WHEN Desc_tipos IS NULL OR Desc_tipos = '' THEN 'NA' ELSE Desc_tipos END AS Desc_tipos,

            -- Nme_tipo_carta / Desc_detalhe_tipo_carta: Planeswalker é tipo
            -- isolado; "—" (em dash) separa tipo principal de subtipo
            -- descritivo. As duas colunas saem da mesma origem.
            CASE
                WHEN Nme_tipo_carta IS NULL THEN NULL
                WHEN lower(Nme_tipo_carta) LIKE '%planeswalker%' THEN 'Planeswalker'
                WHEN instr(Nme_tipo_carta, '—') > 0 THEN trim(split(Nme_tipo_carta, '—', 2)[0])
                ELSE trim(Nme_tipo_carta)
            END AS Nme_tipo_carta,
            CASE
                WHEN Nme_tipo_carta IS NULL THEN NULL
                WHEN lower(Nme_tipo_carta) LIKE '%planeswalker%' THEN Nme_tipo_carta
                WHEN instr(Nme_tipo_carta, '—') > 0 THEN trim(split(Nme_tipo_carta, '—', 2)[1])
                ELSE 'NA'
            END AS Desc_detalhe_tipo_carta,

            to_timestamp(Dt_ingestao) AS Dt_ingestao
        FROM _cards_stage1
    """)

    # Estágio 3: Cod_cores/Desc_subtipos colorless-default (pós-limpeza) e
    # eliminação de "(" ")" "{" "}" do dado Silver (pedido do usuário - esses
    # caracteres sinalizam dado ainda não transformado). \\{ \\} \\( \\) no
    # literal SQL pelo mesmo motivo do Estágio 2 (Spark desfaz backslash
    # simples antes de escape não reconhecido).
    spark.sql(r"""
        CREATE OR REPLACE TEMP VIEW _cards_stage3 AS
        SELECT
            * EXCEPT (Cod_cores, Desc_subtipos, Desc_carta, Desc_custo_mana,
                      Desc_carta_original, Desc_legalidades, Desc_nomes_estrangeiros),

            CASE WHEN Cod_cores IS NULL OR Cod_cores = '' THEN 'Colorless' ELSE Cod_cores END AS Cod_cores,
            CASE WHEN Desc_subtipos IS NULL OR Desc_subtipos = '' THEN 'NA' ELSE Desc_subtipos END AS Desc_subtipos,

            -- Desc_carta: substituições nomeadas pros símbolos de mana mais
            -- comuns (mais legível que colchete genérico), seguidas de dois
            -- catch-alls genéricos: qualquer "{...}" restante (custo
            -- numérico, mana híbrida {W/U}, phyrexiana {W/P}, loyalty
            -- {+1}/{-1} - fora da lista nomeada) e qualquer "(...)" (texto
            -- de lembrete). ponytail: não trata "{" ou "(" aninhados dentro
            -- do mesmo tipo (não ocorre em texto de carta real da Scryfall).
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(
            regexp_replace(Desc_carta, '\\{W\\}', '[White]'),
                                '\\{U\\}', '[Blue]'),
                                '\\{B\\}', '[Black]'),
                                '\\{R\\}', '[Red]'),
                                '\\{G\\}', '[Green]'),
                                '\\{C\\}', '[Colorless]'),
                                '\\{X\\}', '[X]'),
                                '\\{T\\}', '[Tap]'),
                                '\\{Q\\}', '[Untap]'),
                                '\\{S\\}', '[Snow]'),
                                '\\{E\\}', '[Energy]'),
                                '\\{([^}]*)\\}', '[$1]'),
                                '\\(([^)]*)\\)', '[$1]') AS Desc_carta,

            -- Desc_custo_mana: notação puramente simbólica (ex.: "{2}{U}{U}") -
            -- só o catch-all genérico já resolve, sem precisar da lista nomeada.
            regexp_replace(
            regexp_replace(Desc_custo_mana, '\\{([^}]*)\\}', '[$1]'),
                                             '\\(([^)]*)\\)', '[$1]') AS Desc_custo_mana,

            -- Desc_carta_original: texto pré-errata, mesma notação de Desc_carta.
            regexp_replace(
            regexp_replace(Desc_carta_original, '\\{([^}]*)\\}', '[$1]'),
                                                  '\\(([^)]*)\\)', '[$1]') AS Desc_carta_original,

            -- Desc_legalidades: dict serializado (json.dumps) vindo direto da
            -- Bronze - chaves de dict viram colchete pela mesma regra.
            regexp_replace(
            regexp_replace(Desc_legalidades, '\\{([^}]*)\\}', '[$1]'),
                                              '\\(([^)]*)\\)', '[$1]') AS Desc_legalidades,

            -- Desc_nomes_estrangeiros: lista de dicts serializada (um dict por
            -- idioma, sem aninhamento) - mesma regra.
            regexp_replace(
            regexp_replace(Desc_nomes_estrangeiros, '\\{([^}]*)\\}', '[$1]'),
                                                      '\\(([^)]*)\\)', '[$1]') AS Desc_nomes_estrangeiros
        FROM _cards_stage2
    """)

    # Estágio 4: Nme_categoria_cor e Qtd_cores (derivados de Cod_cores e
    # Desc_custo_mana já resolvidos nos estágios anteriores). Qtd_cores conta
    # letras WUBRG restantes - funciona igual antes ou depois da conversão de
    # chave pra colchete no Estágio 3 (a letra em si não muda de posição).
    df_silver = spark.sql("""
        SELECT
            *,
            CASE
                WHEN Cod_cores = 'Colorless' THEN 'Colorless'
                WHEN size(split(Cod_cores, ',')) = 1 THEN 'Mono'
                WHEN size(split(Cod_cores, ',')) = 2 THEN 'Dual Color'
                WHEN size(split(Cod_cores, ',')) >= 3 THEN 'Multicolor'
                ELSE 'Mono'
            END AS Nme_categoria_cor,
            CASE
                WHEN Desc_custo_mana IS NULL OR Desc_custo_mana = 'NA' THEN 0
                ELSE length(regexp_replace(upper(Desc_custo_mana), '[^WUBRG]', ''))
            END AS Qtd_cores
        FROM _cards_stage3
    """)

    logger.info(f"Transformação Cartas concluída: {df_silver.count()} registros")
    return df_silver

def attach_prices(df_cards, df_prices):
    """
    Junta preços (Bronze card_prices) aos cards (Silver, já transformados) pelo
    NOME normalizado - única correspondência real disponível, já que o preço
    vem da Scryfall por nome, sem granularidade de impressão. Uma linha de
    preço se propaga (fan-out) para todas as impressões (Id_carta) do nome.

    lower(trim(...)) em vez de initcap(trim(...)): a comparação só precisa de
    igualdade case-insensitive, não de capitalização de exibição - initcap
    capitaliza após qualquer caractere não-alfabético, então "Urza's Saga"
    virava "Urza'S Saga" e quebrava o match de nomes com apóstrofo.

    LEFT JOIN: card sem preço encontrado continua aparecendo (preço nulo) em
    vez de sumir da Silver. Sem coalesce para 0.0 nas colunas de preço: nulo
    aqui significa "sem dado", não "vale zero".

    Dt_ingestao_preco entra na chave de merge (junto com Id_carta) na hora de
    salvar: a Bronze card_prices guarda só o preço mais recente por nome (merge
    upsert por nome, sem histórico), mas como esse "mais recente" muda de data
    a cada dia, cada execução acrescenta uma nova linha na Silver em vez de
    sobrescrever - é assim que o histórico diário de preço (usado pelos
    relatórios Gold de performance/volatilidade) se acumula.

    Ano_ingestao_preco/Mes_ingestao_preco (#136): derivados de
    Dt_ingestao_preco só para uso como partition_cols na gravação (ver
    save_silver_table abaixo) - não mudam grain, key_column nem esta regra de
    join, só dão ao Delta uma coluna de partição fisicamente presente.
    """
    if df_cards is None:
        return None

    # extract_from_bronze devolve None (em vez de lançar) quando o EXTRACT da
    # Bronze card_prices falha - antes isso caía no mesmo "return df_cards" do
    # caso normal, e a Silver salvava com sucesso sem Vlr_usd/Vlr_eur/Vlr_tix/
    # Dt_ingestao_preco, silenciosamente. Falhar aqui, alto, com a causa no
    # log do EXTRACT logo acima.
    if df_prices is None:
        raise RuntimeError(
            "attach_prices: extract_from_bronze('card_prices') retornou "
            "None. Veja a mensagem 'Erro no EXTRACT da Bronze' no log acima para "
            "a causa - sem isso a Silver salvaria cards sem colunas de preço."
        )

    logger = logging.getLogger(__name__)
    logger.info("Anexando preços (Bronze card_prices) aos cards por nome...")

    df_cards.createOrReplaceTempView("_cards_silver")

    # Padronização de nomes da Bronze card_prices crua (name/usd/eur/tix/
    # ingestion_timestamp) para os nomes PT-BR que a junção abaixo espera -
    # mesma lacuna do Estágio 0 de transform_cards_silver, aqui do lado do preço.
    df_prices.createOrReplaceTempView("_prices_bronze_raw")
    spark.sql("""
        CREATE OR REPLACE TEMP VIEW _prices_bronze AS
        SELECT
            name AS Nme_carta,
            usd AS Vlr_usd,
            eur AS Vlr_eur,
            tix AS Vlr_tix,
            ingestion_timestamp AS Dt_ingestao
        FROM _prices_bronze_raw
    """)

    df_final = spark.sql("""
        SELECT
            cards.*,
            cast(prices.Vlr_usd AS float) AS Vlr_usd,
            cast(prices.Vlr_eur AS float) AS Vlr_eur,
            cast(prices.Vlr_tix AS float) AS Vlr_tix,
            cast(prices.Dt_ingestao AS date) AS Dt_ingestao_preco,
            year(cast(prices.Dt_ingestao AS date)) AS Ano_ingestao_preco,
            month(cast(prices.Dt_ingestao AS date)) AS Mes_ingestao_preco
        FROM _cards_silver cards
        LEFT JOIN _prices_bronze prices
            ON lower(trim(cards.Nme_carta)) = lower(trim(prices.Nme_carta))
    """)

    logger.info(f"Junção com preços concluída: {df_final.count()} registros")
    return df_final

def _resolve_id_chain(direct_map):
    """
    Segue a cadeia de merges old_scryfall_id -> new_scryfall_id até o id final
    (A mergeou em B, B mergeou em C -> A resolve pra C). Puro Python sobre um
    dict pequeno (histórico de migrações da Scryfall, não dado de carta) - sem
    exigir SQL recursivo, que esta versão do Spark não suporta via CTE.
    Testado isoladamente em test_migration_chain.py.
    """
    resolved = {}
    for start in direct_map:
        current = start
        seen = {start}
        hops = 0
        while current in direct_map and hops < 10:
            nxt = direct_map[current]
            if nxt in seen:
                # ciclo (não deveria acontecer em dado real da Scryfall) - para
                # na melhor resolução encontrada em vez de girar pra sempre
                break
            current = nxt
            seen.add(current)
            hops += 1
        resolved[start] = current
    return resolved

def attach_canonical_id(df_cards, df_migrations):
    """
    AUD-20 (#135): resolve a cadeia de migrações de scryfall_id (merge) da
    Scryfall (Bronze migrations) e anexa Id_scryfall_canonico - o id final
    após seguir merges sucessivos, pra Gold agrupar/janelar por ele em vez do
    Id_carta bruto (cuja história pode ter sido fragmentada por uma migração
    no meio da janela de análise). Não reatribui Id_carta (identificador da
    própria impressão, preservado como veio da fonte - restrição do prompt
    de "não alterar IDs/códigos") - só adiciona uma coluna nova.

    Estratégia 'delete' fica fora do mapa de resolução (sem new_scryfall_id,
    não há pra onde apontar) - cards dessas migrações mantêm
    Id_scryfall_canonico = Id_carta, único comportamento possível sem inventar
    um id que a Scryfall não forneceu.

    df_migrations None (Bronze migrations indisponível): degrada pra
    Id_scryfall_canonico = Id_carta em vez de falhar a tabela inteira -
    diferente de attach_prices, migrations é dado de apoio (a fato de cartas
    continua válida sem ela, só sem resolução de migração), não uma coluna de
    negócio esperada em todo run.
    """
    logger = logging.getLogger(__name__)

    if df_migrations is None:
        logger.warning(
            "Bronze migrations indisponível - Id_scryfall_canonico = Id_carta "
            "(sem resolução de migração, ver #135)."
        )
        return df_cards.withColumn("Id_scryfall_canonico", col("Id_carta"))

    merge_rows = (
        df_migrations
        .filter("migration_strategy = 'merge' AND new_scryfall_id IS NOT NULL")
        .select("old_scryfall_id", "new_scryfall_id")
        .distinct()
        .collect()
    )
    direct_map = {r["old_scryfall_id"]: r["new_scryfall_id"] for r in merge_rows}
    resolved_map = _resolve_id_chain(direct_map)

    if not resolved_map:
        return df_cards.withColumn("Id_scryfall_canonico", col("Id_carta"))

    df_map = spark.createDataFrame(
        list(resolved_map.items()), ["_old_scryfall_id", "_canonical_scryfall_id"]
    )
    df_map.createOrReplaceTempView("_migration_map")
    df_cards.createOrReplaceTempView("_cards_pre_canonical")

    df_result = spark.sql("""
        SELECT
            cards.*,
            coalesce(mig._canonical_scryfall_id, cards.Id_carta) AS Id_scryfall_canonico
        FROM _cards_pre_canonical cards
        LEFT JOIN _migration_map mig
            ON cards.Id_carta = mig._old_scryfall_id
    """)
    logger.info(f"Id_scryfall_canonico resolvido para {len(resolved_map)} ids migrados.")
    return df_result

# =============================================================================
# CONFIGURAÇÃO
# =============================================================================

# Configuração manual. catalog_name vem do mesmo secret que a Bronze usa
# (get_secret("catalog_name")).
config = create_manual_config(get_secret("catalog_name"), get_secret("s3_bucket"))

# Setup Unity Catalog
setup_unity_catalog(config['catalog_name'], config['schema_silver'])

In [ ]:
# =============================================================================
# PROCESSAMENTO USANDO SILVER_UTILS
# =============================================================================
# Criar processor - #116: TB_FATO_CARTAS (Fato, ver docstring da célula anterior)
processor = SilverTableProcessor("TB_FATO_CARTAS", config)

# Extração da Bronze (cards) e transformação específica.
# Nomes reais das tabelas Bronze (cards/card_prices/migrations, minúsculo).
df_cards_bronze = processor.extract_from_bronze("cards")
df_cards_silver = processor.transform_data(df_cards_bronze, transform_cards_silver)

# Extração da Bronze (preços) e junção por nome - ver docstring de attach_prices
df_prices_bronze = processor.extract_from_bronze("card_prices")
df_silver = attach_prices(df_cards_silver, df_prices_bronze)

# Extração da Bronze (migrations) e resolução de Id_scryfall_canonico - AUD-20
# (#135). Ao contrário do card_prices acima, uma falha aqui não derruba a
# tabela inteira: migrations é dado de apoio (ver docstring de
# attach_canonical_id) - loga e degrada pra Id_scryfall_canonico = Id_carta.
try:
    df_migrations_bronze = processor.extract_from_bronze("migrations")
except Exception as e:
    logging.getLogger(__name__).warning(
        f"Falha ao extrair Bronze migrations - Id_scryfall_canonico não será resolvido: {e}"
    )
    df_migrations_bronze = None

df_silver = attach_canonical_id(df_silver, df_migrations_bronze)

# Salvar na Silver com particionamento e merge incremental por Id_carta +
# Dt_ingestao_preco.
# Id_carta é o identificador único por impressão (já é a própria chave de
# merge na Bronze) - Nme_carta + Cod_colecao colapsava variantes de arte
# distintas dentro da mesma coleção (showcase, borderless, alt-art) em uma
# única linha na Silver (AUD-10).
# Dt_ingestao_preco entra na chave para não sobrescrever o histórico diário
# de preço a cada execução: a Bronze card_prices só guarda o preço mais
# recente por nome, mas esse "mais recente" muda de data a cada dia, então
# cada execução acrescenta uma nova linha em vez de sobrescrever a anterior
# (ver attach_prices). Cards sem preço (Dt_ingestao_preco nulo) continuam
# tendo uma única linha, como antes. save_to_silver sinaliza esta chave
# (Id_carta, Dt_ingestao_preco) na própria tabela via COMMENT ON TABLE, e
# tenta best-effort uma constraint PRIMARY KEY (degrada pro comentário se
# falhar - Dt_ingestao_preco pode ser NULO, o que UC não permite em PK).
# order_by_col=Dt_ingestao: se o lote tiver mais de uma linha para a mesma
# chave (reprocessamento), mantém a linha da ingestão mais recente em vez de
# uma linha arbitrária (AUD-09)
#
# partition_cols (#136): Ano_ingestao_preco/Mes_ingestao_preco - alinham
# partição física com o padrão de acesso da Gold (janela de ~400 dias sobre
# Dt_ingestao_preco), permitindo pruning de partição em vez de full-scan.
processor.save_silver_table(
    df_silver,
    partition_cols=["Ano_ingestao_preco", "Mes_ingestao_preco"],
    key_column=["Id_carta", "Dt_ingestao_preco"],
    order_by_col="Dt_ingestao"
)

# =============================================================================
# VALIDAÇÃO E LOGS
# =============================================================================
if df_silver:
    print(f"Processamento concluído com sucesso!")
    print(f"Registros processados: {df_silver.count()}")
    print(f"Colunas finais: {df_silver.columns}")
else:
    print("Falha no processamento - DataFrame vazio")
